# Strategic Analysis of Superstore Performance

A full business intelligence report covering sales trends, geographic performance, product profitability and discount strategy.

## 1. Data Scoping and Preparation

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact, Dropdown, IntSlider
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('Sample - Superstore.csv', encoding='latin-1')

print("Dataset shape:", df.shape)
print()
print("Columns:", df.columns.tolist())

Dataset shape: (9994, 21)

Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [3]:
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994 non-null   float64
 18  Quantity       9994

,Row ID,Postal Code,Sales,Quantity,Discount,Profit
count,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,4997.500000,55190.379428,229.858001,3.789574,0.156203,28.656896
std,2885.163629,32063.693350,623.245101,2.225110,0.206452,234.260108
min,1.000000,1040.000000,0.444000,1.000000,0.000000,-6599.978000
25%,2499.250000,23223.000000,17.280000,2.000000,0.000000,1.728750
50%,4997.500000,56430.500000,54.490000,3.000000,0.200000,8.666500
75%,7495.750000,90008.000000,209.940000,5.000000,0.200000,29.364000
max,9994.000000,99301.000000,22638.480000,14.000000,0.800000,8399.976000


### Missing values and duplicates

The dataset has no missing values and no duplicate rows, so no imputation or row removal is needed. The Postal Code column is already an integer.

In [4]:
print("Duplicate rows:", df.duplicated().sum())
print()
print("Missing values per column:")
print(df.isnull().sum())

Duplicate rows: 0

Missing values per column:
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64


### Date conversion

Order Date and Ship Date are stored as strings. Converting them to datetime objects is required for any time-series work.

In [5]:
for col in ['Order Date', 'Ship Date']:
    df[col] = pd.to_datetime(df[col])

print("Data types after conversion:")
print(df[['Order Date', 'Ship Date']].dtypes)

Data types after conversion:
Order Date    datetime64[us]
Ship Date     datetime64[us]
dtype: object


### Feature engineering

Three derived columns make downstream analysis simpler: Profit Margin expresses profit as a percentage of sales; Order Year and Order Month allow time-based grouping.

In [6]:
df['Profit Margin'] = (df['Profit'] / df['Sales']) * 100
df['Order Year']    = df['Order Date'].dt.year
df['Order Month']   = df['Order Date'].dt.month
df['Order Month-Year'] = df['Order Date'].dt.to_period('M')

print("New features created:")
print(df[['Sales', 'Profit', 'Profit Margin', 'Order Year', 'Order Month']].head())

New features created:
      Sales    Profit  Profit Margin  Order Year  Order Month
0  261.9600   41.9136          16.00        2016           11
1  731.9400  219.5820          30.00        2016           11
2   14.6200    6.8714          47.00        2016            6
3  957.5775 -383.0310         -40.00        2015           10
4   22.3680    2.5164          11.25        2015           10


## 2. Deep-Dive Exploratory Analysis

### Time-series trend analysis

The chart below shows total monthly sales. Use the dropdown to filter by product category and look for seasonality or year-over-year growth patterns.

In [ ]:
monthly_sales = df.groupby(['Order Month-Year', 'Category'])['Sales'].sum().reset_index()
monthly_sales['Date'] = monthly_sales['Order Month-Year'].dt.to_timestamp()

def plot_monthly_sales(category='All'):
    plt.figure(figsize=(12, 6))
    if category == 'All':
        total = df.groupby('Order Month-Year')['Sales'].sum()
        plt.plot(total.index.to_timestamp(), total.values,
                 marker='o', linewidth=2, markersize=4, color='steelblue')
        plt.title('Monthly Sales Trend - All Categories', fontsize=14, fontweight='bold')
    else:
        data = monthly_sales[monthly_sales['Category'] == category]
        plt.plot(data['Date'], data['Sales'],
                 marker='o', linewidth=2, markersize=4, color='steelblue')
        plt.title(f'Monthly Sales Trend - {category}', fontsize=14, fontweight='bold')

    plt.xlabel('Date', fontsize=11)
    plt.ylabel('Sales ($)', fontsize=11)
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

categories = ['All'] + list(df['Category'].unique())
interact(plot_monthly_sales, category=Dropdown(options=categories, value='All', description='Category:'));

### Geographic sales performance

A horizontal bar chart sorted by total sales lets us see at a glance which states drive revenue. The slider controls how many states are shown.

In [ ]:
state_sales = df.groupby('State')['Sales'].sum().sort_values(ascending=True)

def plot_top_states(top_n=10):
    top = state_sales.tail(top_n)
    plt.figure(figsize=(12, max(6, top_n * 0.4)))
    bars = plt.barh(range(len(top)), top.values, color='steelblue')
    plt.yticks(range(len(top)), top.index)
    plt.xlabel('Total Sales ($)', fontsize=11)
    plt.ylabel('State', fontsize=11)
    plt.title(f'Top {top_n} States by Sales Performance', fontsize=14, fontweight='bold')
    for i, (state, val) in enumerate(top.items()):
        plt.text(val + top.values.max() * 0.01, i, f'${val:,.0f}', va='center', fontsize=9)
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f"Total states in dataset: {len(state_sales)}")
    print(f"Top {top_n} states account for: ${top.sum():,.0f} in sales")

interact(plot_top_states, top_n=IntSlider(min=5, max=25, value=10, description='Top N States:'));

## 3. Communicating Insights

### Top 10 most profitable products

The bar chart below is suitable for an executive summary. Each bar is annotated with its exact profit value so the reader does not need to read the axis precisely.

In [ ]:
product_profit = df.groupby('Product Name')['Profit'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 8))
ax = sns.barplot(x=product_profit.values, y=product_profit.index,
                 palette='viridis', orient='h')

plt.title('Top 10 Most Profitable Products', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Total Profit ($)', fontsize=11, fontweight='bold')
plt.ylabel('Product Name', fontsize=11, fontweight='bold')

for i, (prod, profit) in enumerate(product_profit.items()):
    ax.text(profit + product_profit.values.max() * 0.01, i,
            f'${profit:,.0f}', va='center', fontsize=9, fontweight='bold')

plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Most profitable product: ${product_profit.iloc[0]:,.0f}")
print(f"Top 10 combined profit: ${product_profit.sum():,.0f}")
print(f"Average profit per product in top 10: ${product_profit.mean():,.0f}")

### Discount vs profit scatter plot

Each point is one transaction. The red dashed line is the overall regression trend. The horizontal line at zero marks the break-even point. Points below it represent losses.

In [ ]:
plt.figure(figsize=(13, 7))

sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category', alpha=0.5, s=40)
sns.regplot(data=df, x='Discount', y='Profit', scatter=False,
            color='red', line_kws={'linewidth': 2, 'linestyle': '--'})

plt.axhline(y=0, color='black', linestyle='-', alpha=0.3, linewidth=1)
plt.text(0.01, 30, 'Break-even line', fontsize=9, alpha=0.6)

plt.title('Discount Strategy: Impact on Profitability by Category',
          fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Discount Rate', fontsize=11, fontweight='bold')
plt.ylabel('Profit ($)', fontsize=11, fontweight='bold')
plt.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

high_disc = df[df['Discount'] > 0.2]
print(f"Transactions with discount above 20%: {len(high_disc):,}")
print(f"Average profit for those transactions: ${high_disc['Profit'].mean():.2f}")
print(f"Share resulting in a loss: {(high_disc['Profit'] < 0).mean() * 100:.1f}%")
print()
print("Category breakdown at discount above 20%:")
for cat in df['Category'].unique():
    subset = df[(df['Category'] == cat) & (df['Discount'] > 0.2)]
    if len(subset):
        print(f"  {cat}: avg profit = ${subset['Profit'].mean():.2f}")

## 4. Methodology and Tooling Review

**Matplotlib** gives fine-grained control over every visual element and integrates cleanly with ipywidgets for interactive charts. It requires more code to produce polished output but is the right tool when precise layout, custom annotations, or dynamic widgets are needed.

**Seaborn** produces clean, publication-ready charts with less code. Statistical overlays like regression lines are a single function call. The trade-off is less direct control over individual elements.

In this notebook the split is: Matplotlib for interactive exploration (time-series, geographic ranking) and Seaborn for communicative charts aimed at stakeholders (product profitability, discount analysis).

In [ ]:
import time

start = time.time()
fig, _ = plt.subplots()
plt.plot(df.groupby('Order Year')['Sales'].sum())
plt.close(fig)
t_mpl = time.time() - start

start = time.time()
fig, _ = plt.subplots()
sns.lineplot(data=df.groupby('Order Year')['Sales'].sum().reset_index(),
             x='Order Year', y='Sales')
plt.close(fig)
t_sns = time.time() - start

print(f"Matplotlib basic plot: {t_mpl:.4f}s")
print(f"Seaborn equivalent:    {t_sns:.4f}s")

## 5. Final Deliverable

### Interactive dashboard

Four charts in a 2x2 layout give a quick overview of the most important dimensions of the dataset: sales over time, revenue by category, top states, and the discount-profit relationship.

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 11))

# Monthly sales trend
monthly_total = df.groupby('Order Month-Year')['Sales'].sum()
ax1.plot(monthly_total.index.to_timestamp(), monthly_total.values,
         marker='o', markersize=3, linewidth=1.5, color='steelblue')
ax1.set_title('Monthly Sales Trend', fontweight='bold')
ax1.set_xlabel('Date')
ax1.set_ylabel('Sales ($)')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True, alpha=0.3)

# Category performance
cat_sales = df.groupby('Category')['Sales'].sum()
ax2.bar(cat_sales.index, cat_sales.values, color=['#4C72B0', '#DD8452', '#55A868'])
ax2.set_title('Sales by Category', fontweight='bold')
ax2.set_ylabel('Sales ($)')
ax2.grid(axis='y', alpha=0.3)

# Top 10 states
top10 = state_sales.tail(10)
ax3.barh(range(len(top10)), top10.values, color='steelblue')
ax3.set_yticks(range(len(top10)))
ax3.set_yticklabels(top10.index)
ax3.set_title('Top 10 States by Sales', fontweight='bold')
ax3.set_xlabel('Sales ($)')
ax3.grid(axis='x', alpha=0.3)

# Discount vs profit
for cat in df['Category'].unique():
    subset = df[df['Category'] == cat]
    ax4.scatter(subset['Discount'], subset['Profit'], label=cat, alpha=0.4, s=20)
ax4.axhline(y=0, color='black', linestyle='--', alpha=0.5)
ax4.set_xlabel('Discount')
ax4.set_ylabel('Profit ($)')
ax4.set_title('Discount vs Profit by Category', fontweight='bold')
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)

plt.suptitle('Superstore Performance Dashboard', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Outlier annotation on the discount-profit chart

The three most and three least profitable individual transactions are labelled so patterns around extreme outcomes are immediately visible.

In [ ]:
plt.figure(figsize=(12, 7))
sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category', alpha=0.5, s=40)

top3    = df.nlargest(3, 'Profit')
bottom3 = df.nsmallest(3, 'Profit')

for _, row in top3.iterrows():
    plt.annotate(f'${row["Profit"]:.0f}',
                 xy=(row['Discount'], row['Profit']),
                 xytext=(8, 4), textcoords='offset points',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='#55A868', alpha=0.8),
                 arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'),
                 fontsize=8)

for _, row in bottom3.iterrows():
    plt.annotate(f'${row["Profit"]:.0f}',
                 xy=(row['Discount'], row['Profit']),
                 xytext=(8, -12), textcoords='offset points',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='#DD8452', alpha=0.8),
                 arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'),
                 fontsize=8)

plt.axhline(y=0, color='black', linestyle='--', alpha=0.4)
plt.title('Discount vs Profit with Outlier Identification', fontsize=13, fontweight='bold')
plt.xlabel('Discount Rate', fontsize=11)
plt.ylabel('Profit ($)', fontsize=11)
plt.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Plotly interactive version

Plotly adds built-in zoom, pan and hover tooltips without any extra widget code. The downside is a larger output file size and a dependency on a browser renderer. For notebooks shared as HTML exports or on platforms like Voila, Plotly works well. For local Jupyter workflows where file size matters, Matplotlib with ipywidgets is usually the better choice.

In [ ]:
try:
    import plotly.express as px

    fig = px.scatter(df, x='Discount', y='Profit', color='Category',
                     hover_data=['Product Name', 'Sales'],
                     trendline='ols',
                     title='Interactive Discount vs Profit Analysis (Plotly)')
    fig.show()
except ImportError:
    print("Plotly is not installed in this environment.")
    print("Run: pip install plotly statsmodels")

## Executive Summary

The analysis covers 9,994 transactions across three product categories and 49 states from 2014 to 2017.

Key findings:

- Sales grow year over year with a consistent Q4 peak, driven mainly by Technology and Office Supplies. Furniture shows the flattest growth.

- California, New York and Texas account for a disproportionate share of revenue. Several mid-tier states show high sales volume but lower-than-average margins, which warrants a pricing review.

- The top 10 products by profit are concentrated in Technology (copiers, phones). A small number of products generate the bulk of profit, so protecting their margins is a priority.

- Discounts above 20% consistently produce losses across all three categories. Furniture is the most sensitive: average profit at high discount levels is negative. A hard ceiling of 20% on standard discounts with a mandatory approval process for exceptions would materially improve overall margins.

Recommended action: review Furniture pricing and discount policy first. Technology upsell opportunities in high-performing states should be explored next.

In [ ]:
total_sales  = df['Sales'].sum()
total_profit = df['Profit'].sum()
margin       = (total_profit / total_sales) * 100

top_state       = state_sales.index[-1]
top_state_sales = state_sales.iloc[-1]
top5_share      = (state_sales.tail(5).sum() / total_sales) * 100

top_cat         = df.groupby('Category')['Sales'].sum().idxmax()
loss_rate       = (df[df['Discount'] > 0.2]['Profit'] < 0).mean() * 100

print("Business performance")
print(f"  Total revenue:      ${total_sales:,.0f}")
print(f"  Total profit:       ${total_profit:,.0f}")
print(f"  Overall margin:     {margin:.1f}%")
print()
print("Geographic performance")
print(f"  Top state:          {top_state} (${top_state_sales:,.0f})")
print(f"  Top 5 states share: {top5_share:.1f}% of total sales")
print()
print("Product performance")
print(f"  Leading category:   {top_cat}")
print(f"  Best product:       {product_profit.index[0]}")
print()
print("Discount strategy")
print(f"  Loss rate at discount > 20%: {loss_rate:.1f}%")
print(f"  Recommended ceiling:         20%")